In [1]:
#@title 1. Install Necessary Libraries
# We'll start by installing all the libraries we need.
# Unsloth will handle the heavy lifting of making the model run fast and efficiently.
# Transformers, peft, accelerate, and trl are all libraries from Hugging Face
# that help with loading models, applying efficient tuning techniques (like LoRA),
# and running the training process.
# --- FIX: Simplified and updated installation commands ---
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "peft" "accelerate" "bitsandbytes" "trl"
# --- ADDED: Install comet_ml for experiment tracking, specifying the required version ---
!pip install "comet_ml>=3.43.2"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-48tqphea/unsloth_17f11fe21c7447fd8f619ca8c1ba4ce4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-48tqphea/unsloth_17f11fe21c7447fd8f619ca8c1ba4ce4
  Resolved https://github.com/unslothai/unsloth.git to commit a78b86e5c9c08b90f53a4ef89e6b9c6860fe66dc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.7 MB/s eta 0:00:

In [2]:
#@title 2. Load the Model and Tokenizer
import torch
from unsloth import FastLanguageModel

# Let's define some key parameters
max_seq_length = 2048 # This is the maximum number of words the model can handle in a single input.
dtype = None # We'll let unsloth decide the best data type.
load_in_4bit = True # This is the magic! Loading in 4-bit drastically reduces memory usage.

# Here, we load our Mistral model using Unsloth's FastLanguageModel.
# This function automatically downloads the model, applies massive speed-ups,
# and prepares it for fine-tuning.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit", # This is a pre-quantized Mistral model, perfect for Colab.
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.1: Fast Mistral patching. Transformers: 4.54.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
#@title 3. Prepare the Model for Fine-Tuning with LoRA
# Now, we need to prepare the model for training.
# We're using a technique called LoRA (Low-Rank Adaptation).
# Think of it like this: instead of training the entire, massive model (which would take too long and too much memory),
# we add tiny, trainable "adapter" layers. We only train these small adapters,
# which is much more efficient. Unsloth makes this incredibly simple.

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # This is a key LoRA parameter. Common values are 8, 16, 32. Higher means more trainable parameters.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # These are the specific layers in the model we'll attach our adapters to.
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True, # This is a memory-saving technique.
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.8.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
#@title 4. Prepare the Instruction Dataset
# This is the most important step. We need to teach the model our specific task.
# The IMDB dataset just has a 'text' (the review) and a 'label' (0 for negative, 1 for positive).
# We need to convert this into an "instruction" format that the model can understand.
# We will format it like a question and answer.

from datasets import load_dataset

# This is the instruction template we'll use.
# It clearly tells the model what to do: "Classify the sentiment..."
# The `### Review:` and `### Sentiment:` sections provide clear structure.
instruction_prompt = """Classify the sentiment of this movie review. Is it positive or negative?

### Review:
{}

### Sentiment:
{}"""

# This function will take a single example from the IMDB dataset
# and format it into our instruction prompt.
def format_row_for_instruction(row):
    # Convert the label (0 or 1) into a human-readable word ("Negative" or "Positive")
    sentiment = "Positive" if row["label"] == 1 else "Negative"
    # Fill in the template with the review text and the sentiment
    formatted_text = instruction_prompt.format(row["text"], sentiment)
    return {"text": formatted_text}

# Load the IMDB dataset from Hugging Face
dataset = load_dataset("imdb", split="train")

# Now, we apply our formatting function to the entire dataset.
# The .map() function is a very efficient way to do this.
dataset = dataset.map(format_row_for_instruction)

# Let's look at one example to see what our formatted data looks like.
print("--- Example of a Formatted Data Row ---")
print(dataset[0]['text'])

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

--- Example of a Formatted Data Row ---
Classify the sentiment of this movie review. Is it positive or negative?

### Review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS

In [16]:
#@title 5. Fine-Tune the Model with the SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
import os
from google.colab import userdata

# --- ADDED: Set up Comet ML environment variables from Colab Secrets ---
os.environ["COMET_API_KEY"] = userdata.get('COMET_API_KEY')
os.environ["COMET_PROJECT_NAME"] = userdata.get('COMET_PROJECT_NAME')
os.environ["COMET_WORKSPACE"] = userdata.get('COMET_WORKSPACE')


# The SFTTrainer (Supervised Fine-Tuning Trainer) is a convenient tool
# from the TRL library that handles the entire training loop for us.

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text", # We tell the trainer to use the 'text' column of our formatted dataset.
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training faster, but we'll keep it simple for now.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # These two combined give an effective batch size of 8.
        warmup_steps = 5,
        # --- FIX: Increased training steps for better model performance ---
        max_steps = 120, # Increased from 60. More steps help the model learn better.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs", # Where to save training outputs.
        # --- ADDED: Tell the trainer to log to Comet ML ---
        report_to = "comet_ml",
    ),
)

# Let's start the training!
print("--- Starting the Fine-Tuning Process ---")
trainer_stats = trainer.train()
print("--- Fine-Tuning Finished! ---")

Unsloth: Tokenizing ["text"]:   0%|          | 0/25000 [00:00<?, ? examples/s]

--- Starting the Fine-Tuning Process ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,000 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)
COMET INFO: An experiment with the same configuration options is already running and will be reused.


Step,Training Loss
1,-0.000100
2,-0.000200
3,0.004500
4,-0.000100
5,-0.000200
6,0.000000
7,-0.000200
8,-0.000300
9,0.000000
10,-0.000100


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


--- Fine-Tuning Finished! ---


In [19]:
#@title 6. Test the Fine-Tuned Model (Inference)
# The training is done, but how do we know if it worked?
# We need to ask our newly fine-tuned model to classify some reviews it has never seen before.
from transformers import TextStreamer

# We need to re-load the base model and tokenizer for inference.
FastLanguageModel.for_inference(model) # This prepares the model for fast predictions.

tokenizer.pad_token = tokenizer.eos_token

# Let's create a couple of test reviews.
review_1 = "This movie was an absolute masterpiece. The acting was superb, the plot was gripping, and the ending was perfect. I was on the edge of my seat the entire time. A must-see!"
review_2 = "What a waste of time. The plot was nonsensical, the characters were flat, and I couldn't wait for it to be over. I would not recommend this to anyone."

# --- FIX: Use a more standard two-step process for preparing the inputs ---
# 1. Format the prompt using the chat template
messages_1 = [
    {"role": "user", "content": f"Classify the sentiment of this movie review. Is it positive or negative?\n\n### Review:\n{review_1}"},
]
prompt_1 = tokenizer.apply_chat_template(messages_1, tokenize = False, add_generation_prompt = True)

messages_2 = [
    {"role": "user", "content": f"Classify the sentiment of this movie review. Is it positive or negative?\n\n### Review:\n{review_2}"},
]
prompt_2 = tokenizer.apply_chat_template(messages_2, tokenize = False, add_generation_prompt = True)


# 2. Tokenize the formatted prompts to get a dictionary of tensors
inputs_1 = tokenizer(prompt_1, return_tensors = "pt").to("cuda")
inputs_2 = tokenizer(prompt_2, return_tensors = "pt").to("cuda")

# Now, let's get the model's predictions.
# --- FIX: Added more robust sampling parameters and explicit stop token ---
outputs_1 = model.generate(
    **inputs_1,
    max_new_tokens = 10,
    do_sample = True,
    temperature = 0.7,
    top_k = 50,
    top_p = 0.95,
    repetition_penalty = 1.1,
    eos_token_id=tokenizer.eos_token_id,
)
decoded_output_1 = tokenizer.decode(outputs_1[0][len(inputs_1['input_ids'][0]):], skip_special_tokens=True)
print("--- Review 1 (Expected: Positive) ---")
print(f"Model Output: '{decoded_output_1.strip()}'")
print("\n" + "="*50 + "\n")


outputs_2 = model.generate(
    **inputs_2,
    max_new_tokens = 10,
    do_sample = True,
    temperature = 0.7,
    top_k = 50,
    top_p = 0.95,
    repetition_penalty = 1.1,
    eos_token_id=tokenizer.eos_token_id,
)
decoded_output_2 = tokenizer.decode(outputs_2[0][len(inputs_2['input_ids'][0]):], skip_special_tokens=True)
print("--- Review 2 (Expected: Negative) ---")
print(f"Model Output: '{decoded_output_2.strip()}'")

# You should now see the model output "### Sentiment: Positive" and "### Sentiment: Negative"


--- Review 1 (Expected: Positive) ---
Model Output: 'Positive posive posive posive posive'


--- Review 2 (Expected: Negative) ---
Model Output: 'Negative.'
